# Insper - Modelagem e Simulação do Mundo Físico

## Projeto 2 - Termodinâmica | Grupo 10

---
## 1. Definição do modelo

**Sistema:** panela de aço inox com 1 L de água, sem tampa, esquentando em um fogão de **indução**. O fogão esquenta o fundo da panela, e a panela esquenta a água.

**Pergunta:** quanto tempo a água demora para ferver em função da potência do fogão?

**Hipóteses:**
- A panela é um cilindro de aço de raio $r$, altura $H$ e espessura $d$ pequena.
- Panela e água têm temperatura uniforme ($T_p$ e $T_a$) e capacidade térmica não desprezível, ou seja, são **2 estoques**.
- Ambiente a $T_{amb} = 25\ °C$ e **nível do mar**: a água ferve a $100\ °C$.
- Não há convecção na água, mas há no ar (como na latinha da Aula 15).
- O fundo da panela está apoiado no fogão e não perde calor para o ambiente.
- Desprezamos a radiação e a evaporação.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

---
## 2. Modelagem

### 2.1 - Esquema físico

<div style="text-align: center;">
    <img src="figuras/esquema_fisico.png" width="600"/>
</div>

- $\dot{Q}_1$: calor gerado pelo fogão de indução no fundo da panela.
- $\dot{Q}_2$: calor da panela para a água (condução).
- $\dot{Q}_3$: calor da panela para o ar pelas paredes (condução + convecção).
- $\dot{Q}_4$: calor da água para o ar pelo topo aberto (convecção).

### 2.2 - Diagrama de estoques e fluxos

<div style="text-align: center;">
    <img src="figuras/diagrama_estoques_fluxos.png" width="700"/>
</div>

- $U_p$: energia interna da panela
- $U_a$: energia interna da água

### 2.3 - Equações diferenciais

**Balanço de energia** (o que entra menos o que sai de cada estoque):

$$
\left\{
    \begin{array}{l}
        \dfrac{dU_p}{dt} = \dot{Q}_1 - \dot{Q}_2 - \dot{Q}_3 \\
        \dfrac{dU_a}{dt} = \dot{Q}_2 - \dot{Q}_4 \\
    \end{array}
\right.
$$

**Fluxos de calor.** $\dot{Q}_1 = \eta \cdot P$ é a potência útil do fogão. Como na latinha da Aula 15, $\dot{Q}_2$, $\dot{Q}_3$ e $\dot{Q}_4$ são escritos com **resistências térmicas**, $\dot{Q} = \dfrac{\Delta T}{R}$:

<div style="text-align: center;">
    <img src="figuras/circuitos_termicos.png" width="620"/>
</div>

- $R_1$: resistência de **convecção** no ar ao redor da panela.
- $R_2$: resistência de **condução** do "meio" da parede até a face externa, ao longo de uma espessura $\frac{d}{2}$.
- $R_3$: resistência de **condução** do "meio" da parede até a face interna, que está em contato com a água, ao longo de uma espessura $\frac{d}{2}$.
- $R_4$: resistência de **convecção** no ar acima da superfície da água (tampa aberta).

$$R_1 = \frac{1}{h_{ar} \cdot A_{lat}} \qquad R_2 = \frac{d/2}{k_{aço} \cdot A_{lat}} \qquad R_3 = \frac{d/2}{k_{aço} \cdot A_{pa}} \qquad R_4 = \frac{1}{h_{ar} \cdot A_{sup}}$$

$$\dot{Q}_2 = \frac{T_p - T_a}{R_3} \qquad \dot{Q}_3 = \frac{T_p - T_{amb}}{R_1 + R_2} \qquad \dot{Q}_4 = \frac{T_a - T_{amb}}{R_4}$$

**Equações para o `odeint`.** Como $\dfrac{dU}{dt} = m \cdot c \cdot \dfrac{dT}{dt}$:

$$
\left\{
    \begin{array}{l}
        \dfrac{dT_p}{dt} = \dfrac{1}{m_p \cdot c_p} \cdot \left( \eta \cdot P - \dfrac{T_p - T_a}{R_3} - \dfrac{T_p - T_{amb}}{R_1 + R_2} \right) \\[4mm]
        \dfrac{dT_a}{dt} = \dfrac{1}{m_a \cdot c_a} \cdot \left( \dfrac{T_p - T_a}{R_3} - \dfrac{T_a - T_{amb}}{R_4} \right) \\
    \end{array}
\right.
$$

**De onde vem cada fórmula** (apostila, Aulas 14 a 16):

| Fórmula | Origem |
|---|---|
| $\dfrac{dU}{dt} = \sum \dot{Q}_{entra} - \sum \dot{Q}_{sai}$ | 1ª Lei da Termodinâmica (sem trabalho) |
| $\dfrac{dU}{dt} = m \cdot c \cdot \dfrac{dT}{dt}$ | calor sensível, $Q = m \cdot c \cdot \Delta T$ |
| $\dot{Q}_1 = \eta \cdot P$ | rendimento: potência útil = rendimento × potência elétrica |
| $R_{cond} = \dfrac{L}{k \cdot A}$ | lei de Fourier (condução) |
| $R_{conv} = \dfrac{1}{h \cdot A}$ | lei de Newton do resfriamento (convecção) |

### 2.4 - Parâmetros

| Parâmetro | Símbolo | Valor | Unidade | Origem |
|---|---|---|---|---|
| raio da panela | $r$ | 0,09 | m | panela comum de 18 cm (**medir**) |
| altura da panela | $H$ | 0,10 | m | panela comum (**medir**) |
| espessura da parede | $d$ | 0,001 | m | chapa de inox típica |
| massa da panela | $m_p$ | 0,8 | kg | panela comum (**pesar**) |
| calor específico do aço | $c_p$ | 500 | J/(kg·K) | tabela |
| condutividade do aço | $k_{aço}$ | 16 | W/(m·K) | tabela |
| massa de água | $m_a$ | 1,0 | kg | 1 L |
| calor específico da água | $c_a$ | 4186 | J/(kg·K) | tabela |
| potência elétrica do fogão | $P$ | 1500 | W | varia no gráfico conclusivo |
| eficiência da indução | $\eta$ | 0,85 | - | típico: 0,84 a 0,90 |
| convecção no ar | $h_{ar}$ | 10 | W/(m²·K) | tabela: convecção natural, 2 a 25 |
| temperatura ambiente | $T_{amb}$ | 25 | °C | condição ambiente |
| temperatura de ebulição | $T_{eb}$ | 100 | °C | nível do mar |

**Áreas:**
- $A_{sup} = \pi r^2$: topo da água.
- $A_{pa} = \pi r^2 + 2 \pi r \cdot h_{agua}$: panela em contato com a água, sendo $h_{agua}$ a altura da água.
- $A_{lat} = 2 \pi r H$: lateral da panela.

Usamos a potência **elétrica** $P$, e não a útil, porque é o valor que se escolhe no fogão e que se conhece no experimento.

Fontes dos valores de tabela: engineeringtoolbox.com e Incropera, *Fundamentos de Transferência de Calor e de Massa*.

In [ ]:
# ---------------- Panela ----------------
r = 0.09          # raio [m]
H = 0.10          # altura [m]
d = 0.001         # espessura da parede [m]
m_p = 0.8         # massa [kg]
c_p = 500         # calor específico do aço [J/(kg.K)]
k_aco = 16        # condutividade térmica do aço [W/(m.K)]

# ---------------- Água ----------------
m_a = 1.0         # massa [kg] (1 L)
c_a = 4186        # calor específico [J/(kg.K)]

# ---------------- Fogão e ambiente ----------------
P = 1500          # potência elétrica [W]
eta = 0.85        # eficiência da indução [-]
h_ar = 10         # coef. de convecção do ar [W/(m2.K)]
T_amb = 25        # temperatura ambiente [°C]
T_eb = 100        # temperatura de ebulição [°C]

# ---------------- Áreas [m2] ----------------
A_sup = np.pi * r**2                      # topo da água
h_agua = (m_a / 1000) / A_sup             # altura da água [m]
A_pa = A_sup + 2 * np.pi * r * h_agua     # panela em contato com a água
A_lat = 2 * np.pi * r * H                 # lateral da panela

# ---------------- Resistências térmicas [K/W] ----------------
R1 = 1 / (h_ar * A_lat)                   # convecção: ar ao redor da panela
R2 = (d / 2) / (k_aco * A_lat)            # condução: meio da parede -> face externa
R3 = (d / 2) / (k_aco * A_pa)             # condução: meio da parede -> face interna (água)
R4 = 1 / (h_ar * A_sup)                   # convecção: ar acima da superfície da água